# Week 6: Building an Intelligent Call Center Pipeline

## Learning Objectives

By the end of this session, you will be able to:
1. **Build an end-to-end AI Services pipeline** (Transcribe → Comprehend → Translate)
2. **Prepare features** from AI Service outputs for ML training
3. **Train XGBoost models** on SageMaker using the Estimator API
4. **Run hyperparameter tuning jobs** to optimize model performance

## Prerequisites

- Completed Week 5 (AWS AI Services fundamentals)
- Watched pre-class videos on SageMaker training, XGBoost algorithms
- Understanding of basic ML concepts (features, labels, training/validation)
- AWS Console access with your student credentials

## SageMaker Studio Environment

You're running this notebook in **Amazon SageMaker Studio**. Today we'll use multiple AWS services together:
- **Amazon Transcribe**: Convert call audio to text
- **Amazon Comprehend**: Analyze sentiment and extract entities from transcripts
- **Amazon Translate**: Handle multilingual calls
- **Amazon SageMaker**: Train and tune ML models at scale

# Section 0: Environment Setup

In this section, we'll:
1. Import required libraries (boto3, pandas, sagemaker SDK)
2. Initialize AWS service clients (Transcribe, Comprehend, Translate, S3, SageMaker)
3. Verify access to our datasets in S3

In [ ]:
# =============================================================================
# IMPORTS AND AWS CLIENT INITIALIZATION
# =============================================================================
# Standard libraries
import boto3                    # AWS SDK for Python
import pandas as pd             # Data manipulation
import numpy as np              # Numerical operations
import json                     # JSON parsing for API responses
from datetime import datetime   # Timestamps for job names and outputs
import time                     # For polling and rate limiting

# SageMaker SDK for training
import sagemaker
from sagemaker import get_execution_role
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter

# =============================================================================
# STUDENT IDENTIFICATION
# =============================================================================
# You will be prompted to enter your student number when you run this cell

STUDENT_NAME = input("Enter your student number (e.g., student1): ").strip()

if not STUDENT_NAME:
    raise ValueError("❌ Please enter your student number!")

print(f"✅ Student: {STUDENT_NAME}")

# =============================================================================
# INITIALIZE AWS CLIENTS
# =============================================================================
# SageMaker notebooks have credentials pre-configured via IAM role

session = boto3.Session()
region = session.region_name

# AI Services clients
transcribe = boto3.client('transcribe', region_name=region)  # Audio → text
comprehend = boto3.client('comprehend', region_name=region)  # NLP analysis
translate = boto3.client('translate', region_name=region)    # Language translation

# Storage and compute
s3 = boto3.client('s3', region_name=region)
sm_session = sagemaker.Session()
role = get_execution_role()

# Account identification
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']

# =============================================================================
# DEFINE BUCKET AND PATHS
# =============================================================================
BUCKET_NAME = f"sagemaker-academy-{account_id}"
DATA_PREFIX = "week6/data"
OUTPUT_PREFIX = f"week6/outputs/{STUDENT_NAME}"  # Each student gets their own folder

print("=" * 60)
print("AWS Environment Setup Complete")
print("=" * 60)
print(f"Student:         {STUDENT_NAME}")
print(f"Region:          {region}")
print(f"Account:         {account_id}")
print(f"S3 Bucket:       {BUCKET_NAME}")
print(f"SageMaker Role:  {role.split('/')[-1]}")
print(f"Data Path:       s3://{BUCKET_NAME}/{DATA_PREFIX}/")
print(f"Output Path:     s3://{BUCKET_NAME}/{OUTPUT_PREFIX}/")
print("=" * 60)

In [ ]:
# =============================================================================
# VERIFY DATA ACCESS
# =============================================================================
# Before we start processing, let's confirm we can access all our datasets

print("Verifying access to datasets in S3...\n")

# Call recording audio files
print("1. Call Recordings (Audio for Transcribe):")
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=f"{DATA_PREFIX}/call_recordings/")
audio_count = 0
for obj in response.get('Contents', []):
    if obj['Key'].endswith('.wav') or obj['Key'].endswith('.mp3'):
        print(f"   [OK] {obj['Key'].split('/')[-1]}")
        audio_count += 1
print(f"   Total: {audio_count} audio files")

# Pre-computed transcripts (fallback)
print("\n2. Pre-Computed Transcripts (Backup):")
try:
    s3.head_object(Bucket=BUCKET_NAME, Key=f"{DATA_PREFIX}/call_transcripts.json")
    print(f"   [OK] call_transcripts.json")
except:
    print(f"   [WARNING] call_transcripts.json not found - will rely on Transcribe")

# Transaction sequences
print("\n3. Transaction Data (for Feature Engineering):")
try:
    s3.head_object(Bucket=BUCKET_NAME, Key=f"{DATA_PREFIX}/transaction_sequences.csv")
    print(f"   [OK] transaction_sequences.csv")
except:
    print(f"   [ERROR] transaction_sequences.csv not found")

print("\n" + "=" * 60)
print("Data verification complete!")
print("=" * 60)

# What Are We Building Today?

## Scenario: Call Center Fraud Detection Pipeline

Imagine you're a data scientist at a **financial services company** with a large call center. Your challenge:

**Problem**: Thousands of customer service calls come in daily. Some involve fraud reports, chargebacks, and disputed transactions. Currently, fraud analysts manually review call recordings - a slow, expensive process.

**Your Solution**: Build an automated ML pipeline that:
1. **Transcribes** call recordings to text
2. **Analyzes** sentiment and extracts key information (entities, topics)
3. **Combines** call insights with transaction history
4. **Predicts** which calls are likely fraud-related

## The Pipeline We'll Build

```
Call Audio Files (WAV)
         ↓
    TRANSCRIBE
         ↓
    Text Transcripts
         ↓
    COMPREHEND → Sentiment, Entities, Key Phrases
         ↓
   (TRANSLATE) → Handle Spanish calls
         ↓
    NLP Features
         ↓
Feature Engineering ← Transaction Data (CSV)
         ↓
  Combined Features
         ↓
    SAGEMAKER XGBoost Training
         ↓
 Hyperparameter Tuning
         ↓
    Optimized Fraud Detection Model
```

## Why This Matters

Instead of manually reviewing every call:
- **Automatically prioritize** high-risk calls for analyst review
- **Reduce investigation time** by highlighting key entities (amounts, accounts, locations)
- **Detect patterns** across calls and transactions that humans might miss
- **Scale** to thousands of calls per day

In [ ]:
# =============================================================================
# LOAD TRANSACTION DATA
# =============================================================================
# This CSV contains transaction sequences linked to call recordings
# We'll use this later to create features for our fraud detection model

tx_key = f"{DATA_PREFIX}/transaction_sequences.csv"

# Download from S3 and load into pandas
response = s3.get_object(Bucket=BUCKET_NAME, Key=tx_key)
transactions_df = pd.read_csv(response['Body'])

print(f"Loaded {len(transactions_df)} transaction records\n")
print("Columns:", list(transactions_df.columns))
print(f"\nUnique customers (calls): {transactions_df['call_id'].nunique()}")
print(f"Fraud cases: {transactions_df['is_fraud'].sum()} / {transactions_df['call_id'].nunique()} calls")

# Preview the data
print("\nFirst few transactions:")
display(transactions_df.head(10))

# Show transaction summary by call
print("\n" + "=" * 60)
print("Transaction Summary by Call:")
print("=" * 60)
summary = transactions_df.groupby('call_id').agg({
    'sequence_id': 'count',
    'amount': ['mean', 'max'],
    'is_fraud': 'first'
}).round(2)
summary.columns = ['num_transactions', 'avg_amount', 'max_amount', 'is_fraud']
display(summary)

# Part 1: Audio Transcription with Amazon Transcribe

Amazon Transcribe converts speech to text using automatic speech recognition (ASR). It's designed for analyzing recorded audio or real-time streaming.

## Key Features

| Feature | Purpose | Use Case |
|---------|---------|----------|
| **Automatic Speech Recognition** | Convert audio to text | Call center recordings, meetings, podcasts |
| **Speaker Diarization** | Identify different speakers | Separate agent vs customer in calls |
| **Custom Vocabularies** | Improve accuracy for domain terms | Financial jargon, product names |
| **Multiple Languages** | Support 100+ languages | Multilingual customer base |

## Async Job Pattern

Transcribe processes audio files **asynchronously** via jobs:

```python
# 1. Start a transcription job
response = transcribe.start_transcription_job(
    TranscriptionJobName='unique-job-name',  # Must be unique!
    Media={'MediaFileUri': 's3://bucket/audio.wav'},
    MediaFormat='wav',
    LanguageCode='en-US',
    Settings={
        'ShowSpeakerLabels': True,   # Enable speaker diarization
        'MaxSpeakerLabels': 2        # Agent + Customer
    }
)

# 2. Poll for completion
while True:
    status = transcribe.get_transcription_job(
        TranscriptionJobName='unique-job-name'
    )
    if status['TranscriptionJob']['TranscriptionJobStatus'] == 'COMPLETED':
        break
    time.sleep(5)  # Wait 5 seconds before checking again

# 3. Download the transcript from the result URI
transcript_uri = status['TranscriptionJob']['Transcript']['TranscriptFileUri']
```

## Important Notes

- **Job names must be globally unique** - use timestamp to ensure uniqueness
- **Jobs can take 1-3 minutes** for short recordings
- **Timeout handling** - we'll use a fallback if jobs don't complete in time
- **Audio must be in S3** - Transcribe cannot access local files

## Demo: Transcribing a Single Call

Let's transcribe one call recording to understand the process.

In [ ]:
# =============================================================================
# DEMO: TRANSCRIBE A SINGLE CALL RECORDING
# =============================================================================

# For this demo, we'll transcribe call_001.wav
demo_call = "call_001"
demo_audio_key = f"{DATA_PREFIX}/call_recordings/{demo_call}.wav"

# Create a unique job name using timestamp
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
job_name = f"academy-{demo_call}-{timestamp}"

print(f"Starting transcription job: {job_name}")
print(f"Audio file: s3://{BUCKET_NAME}/{demo_audio_key}")
print("=" * 60)

try:
    # Start the transcription job
    response = transcribe.start_transcription_job(
        TranscriptionJobName=job_name,
        Media={'MediaFileUri': f"s3://{BUCKET_NAME}/{demo_audio_key}"},
        MediaFormat='wav',
        LanguageCode='en-US',
        Settings={
            'ShowSpeakerLabels': True,   # Identify different speakers
            'MaxSpeakerLabels': 2        # Agent + Customer
        }
    )
    
    print(f"✓ Job started successfully")
    print(f"  Job Name: {job_name}")
    print(f"  Status: {response['TranscriptionJob']['TranscriptionJobStatus']}")
    
    # Poll for completion (30 second timeout for demo)
    print("\n⏳ Waiting for job to complete (max 30 seconds for demo)...")
    
    start_time = time.time()
    while time.time() - start_time < 30:  # 30 second timeout
        job_response = transcribe.get_transcription_job(TranscriptionJobName=job_name)
        status = job_response['TranscriptionJob']['TranscriptionJobStatus']
        
        if status == 'COMPLETED':
            print(f"\n✓ Transcription completed!")
            
            # Get the transcript URI
            transcript_uri = job_response['TranscriptionJob']['Transcript']['TranscriptFileUri']
            print(f"  Transcript URI: {transcript_uri}")
            
            # Download and parse the transcript JSON
            # The URI is a pre-signed S3 URL, we can download it directly
            import urllib.request
            with urllib.request.urlopen(transcript_uri) as url:
                transcript_json = json.loads(url.read().decode())
            
            # Extract the transcript text
            transcript_text = transcript_json['results']['transcripts'][0]['transcript']
            print(f"\n📝 Transcript Preview:")
            print(transcript_text[:300] + "..." if len(transcript_text) > 300 else transcript_text)
            break
            
        elif status == 'FAILED':
            print(f"\n✗ Job failed: {job_response['TranscriptionJob'].get('FailureReason', 'Unknown')}")
            break
        
        time.sleep(5)  # Check every 5 seconds
    else:
        print(f"\n⏱️ Demo timeout - job still in progress")
        print(f"   In the lab, we'll use longer timeouts or fallback to pre-computed transcripts")
        
except transcribe.exceptions.ConflictException:
    print(f"✓ Job already exists - transcription may have completed previously")
except Exception as e:
    print(f"✗ Error: {str(e)}")

## Lab 1: Transcribe All Call Recordings

Now it's your turn to transcribe all call recordings (call_001 through call_005).

### Your Task

1. **Upload audio files to S3** (if not already there)
2. **Start transcription jobs** for all 5 calls with unique job names
3. **Poll for completion** with a 180-second timeout per job
4. **Parse transcripts** from the result URIs
5. **Fallback handling**: If any jobs timeout or fail, load pre-computed transcripts from JSON

### Expected Output

A dictionary mapping call IDs to transcript text:
```python
{
    'call_001': 'Thank you for calling First National Bank...',
    'call_002': 'Hello, I need to check my account balance...',
    ...
}
```

### Hints

- Use `datetime.now().strftime("%Y%m%d%H%M%S")` for unique timestamps
- Job names must be unique - include call_id AND timestamp
- Poll every 5 seconds, maximum 180 seconds (3 minutes) per job
- If transcript URI download fails, use pre-computed JSON
- The pre-computed transcripts are in: `s3://{BUCKET_NAME}/{DATA_PREFIX}/call_transcripts.json`

In [ ]:
# =============================================================================
# LAB 1: TRANSCRIBE ALL CALL RECORDINGS
# =============================================================================

def wait_for_transcription_job(job_name, timeout=180):
    """
    Poll a Transcribe job until completion or timeout.
    Returns the transcript URI if successful, None if timeout/failure.
    """
    start_time = time.time()
    
    while time.time() - start_time < timeout:
        # YOUR CODE: Get job status using transcribe.get_transcription_job()
        job_response = None  # YOUR CODE
        
        status = None  # YOUR CODE: Extract status from job_response
        
        if status == 'COMPLETED':
            # YOUR CODE: Extract and return transcript URI
            return None  # YOUR CODE: Return the TranscriptFileUri
        elif status == 'FAILED':
            print(f"    Job failed")
            return None
        
        time.sleep(5)  # Poll every 5 seconds
    
    print(f"    Timeout after {timeout} seconds")
    return None


def download_transcript_from_uri(transcript_uri):
    """
    Download and parse transcript JSON from S3 URI.
    Returns the transcript text.
    """
    import urllib.request
    
    # YOUR CODE: Download JSON from URI and extract transcript text
    try:
        with urllib.request.urlopen(transcript_uri) as url:
            transcript_json = None  # YOUR CODE: Parse JSON
            transcript_text = None  # YOUR CODE: Extract text from results
            return transcript_text
    except Exception as e:
        print(f"    Error downloading transcript: {e}")
        return None


def load_fallback_transcripts():
    """
    Load pre-computed transcripts from JSON file as fallback.
    Returns dictionary {call_id: transcript_text}
    """
    try:
        # YOUR CODE: Load call_transcripts.json from S3
        fallback_key = None  # YOUR CODE: f"{DATA_PREFIX}/call_transcripts.json"
        response = None  # YOUR CODE: s3.get_object()
        transcripts_json = None  # YOUR CODE: Parse JSON
        
        # Extract just the transcript text for each call
        fallback_transcripts = {}
        # YOUR CODE: Loop through and extract 'transcript' field
        
        return fallback_transcripts
    except Exception as e:
        print(f"Error loading fallback transcripts: {e}")
        return {}


# Main processing loop
print("Transcribing all call recordings...")
print("=" * 60)

# List of calls to process
call_ids = ['call_001', 'call_002', 'call_003', 'call_004', 'call_005']

transcripts = {}  # Will store {call_id: transcript_text}
failed_calls = []  # Track which calls need fallback

# Generate timestamp for this batch
batch_timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

for call_id in call_ids:
    print(f"\nProcessing {call_id}...")
    
    # YOUR CODE: Define audio file S3 key
    audio_key = None  # YOUR CODE: f"{DATA_PREFIX}/call_recordings/{call_id}.wav"
    
    # YOUR CODE: Create unique job name
    job_name = None  # YOUR CODE: Include call_id and batch_timestamp
    
    try:
        # YOUR CODE: Start transcription job
        response = None  # YOUR CODE: transcribe.start_transcription_job()
        
        print(f"  ✓ Job started: {job_name}")
        
        # YOUR CODE: Wait for completion
        transcript_uri = None  # YOUR CODE: Call wait_for_transcription_job()
        
        if transcript_uri:
            # YOUR CODE: Download and parse transcript
            transcript_text = None  # YOUR CODE: Call download_transcript_from_uri()
            
            if transcript_text:
                transcripts[call_id] = transcript_text
                print(f"  ✓ Transcription successful ({len(transcript_text)} characters)")
            else:
                failed_calls.append(call_id)
        else:
            failed_calls.append(call_id)
            
    except transcribe.exceptions.ConflictException:
        print(f"  ! Job already exists - attempting to retrieve existing result")
        # YOUR CODE: Try to get existing job result
        failed_calls.append(call_id)
    except Exception as e:
        print(f"  ✗ Error: {e}")
        failed_calls.append(call_id)

# Fallback: Load pre-computed transcripts for any failed calls
if failed_calls:
    print(f"\n" + "=" * 60)
    print(f"Loading fallback transcripts for {len(failed_calls)} calls...")
    fallback = load_fallback_transcripts()
    
    for call_id in failed_calls:
        if call_id in fallback:
            transcripts[call_id] = fallback[call_id]
            print(f"  ✓ {call_id}: Loaded from fallback")

print(f"\n" + "=" * 60)
print(f"✅ Transcription complete: {len(transcripts)} calls processed")
print("=" * 60)

In [ ]:
# =============================================================================
# VIEW TRANSCRIPTION RESULTS
# =============================================================================

print("Transcription Results:")
print("=" * 60)

for call_id, transcript in transcripts.items():
    print(f"\n{call_id}:")
    print(f"  Length: {len(transcript)} characters")
    print(f"  Word count: ~{len(transcript.split())} words")
    print(f"  Preview: {transcript[:150]}...")

# Show one complete transcript
print("\n" + "=" * 60)
print("Complete Transcript Example (call_001):")
print("=" * 60)
if 'call_001' in transcripts:
    print(transcripts['call_001'])

# Part 2: Text Analysis with Amazon Comprehend

Now that we have text transcripts, let's extract insights using **Amazon Comprehend** - the same NLP service you used in Week 5.

## What We'll Extract

| API | Feature | Why It Matters for Fraud Detection |
|-----|---------|-------------------------------------|
| `detect_sentiment()` | Emotional tone | Negative sentiment may indicate disputes, fraud claims |
| `detect_entities()` | Names, locations, amounts, dates | Extract transaction details, account numbers |
| `detect_key_phrases()` | Important concepts | Identify topics: "unauthorized charge", "stolen card" |

## Optional: Handling Multilingual Calls

Some customers may call in languages other than English. We can:
1. **Detect language** with `detect_dominant_language()`
2. **Translate to English** with Amazon Translate
3. **Analyze the translated text** with Comprehend

## Demo: Analyzing a Single Transcript

Let's analyze one call transcript to see what insights we can extract.

In [ ]:
# =============================================================================
# DEMO: ANALYZE SINGLE TRANSCRIPT WITH COMPREHEND
# =============================================================================

if 'call_001' in transcripts:
    demo_transcript = transcripts['call_001']
    
    print("Analyzing transcript from call_001...")
    print("=" * 60)
    
    # 1. Detect Sentiment
    sentiment_response = comprehend.detect_sentiment(
        Text=demo_transcript,
        LanguageCode='en'
    )
    
    print("\n1. SENTIMENT ANALYSIS:")
    print(f"   Overall Sentiment: {sentiment_response['Sentiment']}")
    print(f"   Confidence Scores:")
    for sentiment, score in sentiment_response['SentimentScore'].items():
        bar = "█" * int(score * 20)
        print(f"     {sentiment:10s}: {score:.3f} {bar}")
    
    # 2. Detect Entities
    entities_response = comprehend.detect_entities(
        Text=demo_transcript,
        LanguageCode='en'
    )
    
    print(f"\n2. NAMED ENTITIES ({len(entities_response['Entities'])} found):")
    for entity in entities_response['Entities'][:10]:  # Show first 10
        print(f"   {entity['Type']:15s} | {entity['Text']:30s} | Confidence: {entity['Score']:.2f}")
    
    # 3. Detect Key Phrases
    keyphrases_response = comprehend.detect_key_phrases(
        Text=demo_transcript,
        LanguageCode='en'
    )
    
    print(f"\n3. KEY PHRASES ({len(keyphrases_response['KeyPhrases'])} found):")
    for phrase in keyphrases_response['KeyPhrases'][:10]:  # Show first 10
        print(f"   • {phrase['Text']} (score: {phrase['Score']:.2f})")
    
    print("\n" + "=" * 60)
    print("✓ Analysis complete for demo transcript")
    print("=" * 60)

## Lab 2: Analyze All Transcripts with Comprehend

Apply NLP analysis to all transcribed calls to create features for our fraud detection model.

### Your Task

1. **Create analysis function** that calls all three Comprehend APIs
2. **Loop through all transcripts** and analyze each one
3. **Extract key features**:
   - Sentiment label and confidence scores
   - Entity count and entity types
   - Key phrase count
   - Word count from the transcript
4. **Optional**: Detect language and translate if not English (for call_005)
5. **Create DataFrame** with all extracted features

### Expected Output

A DataFrame with columns:
- `call_id`: The call identifier
- `sentiment`: POSITIVE, NEGATIVE, NEUTRAL, or MIXED
- `positive_score`, `negative_score`, `neutral_score`: Confidence scores
- `entity_count`: Number of entities found
- `key_phrase_count`: Number of key phrases
- `word_count`: Words in transcript

### Hints

- Comprehend has a 5000 byte limit - truncate long transcripts if needed
- Use `time.sleep(0.1)` between API calls to avoid throttling
- For word count: `len(transcript.split())`
- Store entity types as a list for later analysis

In [ ]:
# =============================================================================
# LAB 2: ANALYZE ALL TRANSCRIPTS WITH COMPREHEND
# =============================================================================

def analyze_transcript(text):
    """
    Analyze a single transcript with Comprehend.
    Returns dict with sentiment, entities, and key phrases.
    """
    # Truncate if needed (Comprehend 5000 byte limit)
    text = text[:4500] if len(text) > 4500 else text
    
    results = {}
    
    # YOUR CODE: Call detect_sentiment
    sentiment_response = None  # YOUR CODE
    results['sentiment'] = None  # YOUR CODE: Extract sentiment label
    results['sentiment_scores'] = None  # YOUR CODE: Extract scores dict
    
    # YOUR CODE: Call detect_entities
    entities_response = None  # YOUR CODE
    results['entities'] = None  # YOUR CODE: Extract Entities list
    
    # YOUR CODE: Call detect_key_phrases  
    keyphrases_response = None  # YOUR CODE
    results['key_phrases'] = None  # YOUR CODE: Extract KeyPhrases list
    
    return results


# Process all transcripts
print("Analyzing transcripts with Comprehend...")
print("=" * 60)

comprehend_results = []

for call_id, transcript in transcripts.items():
    print(f"\nProcessing {call_id}...")
    
    # YOUR CODE: Analyze the transcript
    analysis = None  # YOUR CODE: Call analyze_transcript()
    
    # YOUR CODE: Extract features into result dictionary
    result = {
        'call_id': call_id,
        'sentiment': None,  # YOUR CODE
        'positive_score': None,  # YOUR CODE: Round to 3 decimals
        'negative_score': None,  # YOUR CODE
        'neutral_score': None,  # YOUR CODE
        'mixed_score': None,  # YOUR CODE
        'entity_count': None,  # YOUR CODE: len(analysis['entities'])
        'entities': None,  # YOUR CODE: List of entity texts
        'entity_types': None,  # YOUR CODE: List of entity types
        'key_phrase_count': None,  # YOUR CODE
        'key_phrases': None,  # YOUR CODE: Top 5 key phrases
        'word_count': None  # YOUR CODE: len(transcript.split())
    }
    
    comprehend_results.append(result)
    print(f"  ✓ {result['sentiment']} (Entities: {result['entity_count']}, Words: {result['word_count']})")
    
    # Small delay to avoid throttling
    time.sleep(0.1)

# Create DataFrame
comprehend_df = pd.DataFrame(comprehend_results)

print(f"\n" + "=" * 60)
print(f"✅ Comprehend analysis complete: {len(comprehend_df)} calls")
print("=" * 60)

In [ ]:
# =============================================================================
# VIEW COMPREHEND RESULTS
# =============================================================================

print("Comprehend Analysis Results:")
print("=" * 60)

# Display the DataFrame
display(comprehend_df)

# Sentiment distribution
print("\nSentiment Distribution:")
print(comprehend_df['sentiment'].value_counts())

# Show detailed analysis for one call
print("\n" + "=" * 60)
print("Detailed Analysis - call_001:")
print("=" * 60)
if len(comprehend_df) > 0:
    sample = comprehend_df[comprehend_df['call_id'] == 'call_001'].iloc[0]
    print(f"Sentiment:      {sample['sentiment']}")
    print(f"Positive Score: {sample['positive_score']}")
    print(f"Negative Score: {sample['negative_score']}")
    print(f"Entities:       {sample['entities']}")
    print(f"Key Phrases:    {sample['key_phrases']}")

# Part 3: Feature Engineering

We now have two rich sources of information:
1. **NLP features from Comprehend**: Sentiment, entities, key phrases from call transcripts
2. **Transaction data**: Historical transaction patterns linked to each call

The next step is to **combine these features** into a single dataset suitable for machine learning.

## Why Feature Engineering Matters

Machine learning models like XGBoost need **numerical features** that capture patterns distinguishing fraud from normal behavior:

| Feature Source | Example Features | Why They Matter |
|----------------|------------------|-----------------|
| **Call NLP** | Sentiment scores, entity count, word count | Fraudulent calls may have negative sentiment, mention specific entity types |
| **Transactions** | Transaction count, average amount, max amount | Fraud patterns: high-value transactions, unusual frequency |
| **Combined** | Cross-feature patterns | Model can learn: "negative sentiment + high avg transaction = fraud risk" |

## The SageMaker XGBoost Data Format

**CRITICAL**: SageMaker's built-in XGBoost requires a specific CSV format:

```
⚠️ REQUIRED FORMAT:
- Target column (is_fraud) must be FIRST column
- No header row
- No index column
- Comma-separated values
```

Example:
```
1,NEGATIVE,0.89,0.05,12,458,5,2567.34,8950.00,0.4
0,POSITIVE,0.12,0.82,8,312,3,124.56,450.00,0.0
```

First column is the label (1=fraud, 0=normal), followed by all features.

## Demo: Transaction Aggregation

Let's start by aggregating transaction data by call_id to create per-call features.

In [ ]:
# =============================================================================
# DEMO: AGGREGATE TRANSACTION DATA BY CALL
# =============================================================================

print("Aggregating transaction data by call_id...")
print("=" * 60)

# We want to create summary features for each call's transaction history
# These features capture spending patterns that may indicate fraud risk

tx_features = transactions_df.groupby('call_id').agg({
    'sequence_id': 'count',                    # Total number of transactions
    'amount': ['mean', 'max', 'std'],          # Amount statistics
    'is_international': 'mean',                # Percentage of international transactions
    'is_online': 'mean',                       # Percentage of online transactions
    'is_fraud': 'first'                        # Label (same for all rows of a call)
}).reset_index()

# Flatten multi-level column names
tx_features.columns = [
    'call_id',
    'transaction_count',
    'avg_amount',
    'max_amount',
    'std_amount',
    'international_pct',
    'online_pct',
    'is_fraud'
]

# Round for readability
tx_features = tx_features.round(2)

print(f"Created transaction features for {len(tx_features)} calls\n")
print("Feature Columns:")
for col in tx_features.columns:
    print(f"  • {col}")

print("\n" + "=" * 60)
print("Sample Transaction Features:")
print("=" * 60)
display(tx_features.head())

# Show fraud vs normal patterns
print("\n" + "=" * 60)
print("Fraud vs Normal Transaction Patterns:")
print("=" * 60)
fraud_summary = tx_features.groupby('is_fraud').agg({
    'transaction_count': 'mean',
    'avg_amount': 'mean',
    'max_amount': 'mean',
    'international_pct': 'mean',
    'online_pct': 'mean'
}).round(2)

fraud_summary.index = ['Normal', 'Fraud']
display(fraud_summary)

print("\n💡 Notice how fraud cases have different patterns:")
print("   - Higher average amounts")
print("   - More international transactions")
print("   - Different online transaction rates")

## Lab 3: Combine Features and Prepare for SageMaker

Create the final dataset for training by combining NLP features from Comprehend with transaction features.

### Your Task

**Step 1: Create NLP Feature DataFrame**
- Extract key features from `comprehend_df`:
  - `call_id`
  - Sentiment scores (positive, negative, neutral)
  - Entity count
  - Word count

**Step 2: Merge with Transaction Features**
- Join NLP features with `tx_features` on `call_id`
- Result should have both NLP and transaction columns

**Step 3: Encode Categorical Features**
- Convert `sentiment` to numeric (map POSITIVE=2, NEUTRAL=1, NEGATIVE=0)

**Step 4: Reorder Columns for XGBoost**
- ⚠️ **CRITICAL**: Move `is_fraud` to FIRST position
- XGBoost requires the target as column 0

**Step 5: Split Train/Validation**
- Use 80/20 split
- Random state for reproducibility

**Step 6: Upload to S3**
- Save as CSV with **no headers** and **no index**
- Upload to: `s3://{BUCKET_NAME}/{OUTPUT_PREFIX}/train.csv`
- Upload to: `s3://{BUCKET_NAME}/{OUTPUT_PREFIX}/validation.csv`

### Expected Output

Two CSV files in S3:
- `train.csv`: 80% of data, ~4 samples (small dataset for demo)
- `validation.csv`: 20% of data, ~1 sample

Format:
```
1,0.05,0.89,0.12,12,458,5,2567.34,8950.00,0.4,0.2
0,0.82,0.12,0.15,8,312,3,124.56,450.00,0.0,0.1
```

### Hints

- Use `pd.merge(left, right, on='call_id')` to join DataFrames
- Sentiment mapping: `{'POSITIVE': 2, 'NEUTRAL': 1, 'NEGATIVE': 0, 'MIXED': 1}`
- Reorder: `cols = ['is_fraud'] + [col for col in df.columns if col != 'is_fraud']`
- Save without headers: `df.to_csv(file, header=False, index=False)`
- Upload with: `s3.upload_file(local_path, BUCKET_NAME, s3_key)`

In [ ]:
# =============================================================================
# LAB 3: COMBINE FEATURES AND PREPARE FOR SAGEMAKER
# =============================================================================

print("Creating combined feature dataset...")
print("=" * 60)

# Step 1: Create NLP features DataFrame from Comprehend results
nlp_features = comprehend_df[[
    'call_id',
    'positive_score',
    'negative_score',
    'neutral_score',
    'entity_count',
    'word_count'
]].copy()

# Step 2: Merge NLP features with transaction features
# YOUR CODE: Use pd.merge() to join on call_id
merged_df = None  # YOUR CODE

print(f"✓ Merged dataset created: {len(merged_df)} rows")
print(f"  Columns: {list(merged_df.columns)}")

# Step 3: Encode categorical features
# YOUR CODE: Map sentiment to numeric values
# POSITIVE=2, NEUTRAL=1, NEGATIVE=0, MIXED=1
sentiment_map = None  # YOUR CODE
# YOUR CODE: Apply mapping if you kept sentiment column (optional)

# Step 4: Select final feature set (drop non-numeric columns)
# Keep only the features we want for training
feature_columns = [
    'is_fraud',           # TARGET (will move to first position next)
    'positive_score',
    'negative_score', 
    'neutral_score',
    'entity_count',
    'word_count',
    'transaction_count',
    'avg_amount',
    'max_amount',
    'std_amount',
    'international_pct',
    'online_pct'
]

# YOUR CODE: Select only these columns
final_df = None  # YOUR CODE: merged_df[feature_columns]

# Step 5: CRITICAL - Reorder columns to put target (is_fraud) FIRST
# YOUR CODE: Reorder so is_fraud is column 0
cols = None  # YOUR CODE: ['is_fraud'] + [col for col in final_df.columns if col != 'is_fraud']
final_df = None  # YOUR CODE: final_df[cols]

print(f"\n✓ Final feature set prepared")
print(f"  Shape: {final_df.shape}")
print(f"  First column (target): {final_df.columns[0]}")
print(f"  Feature columns: {final_df.columns[1:].tolist()}")

# Step 6: Split into train/validation
from sklearn.model_selection import train_test_split

# YOUR CODE: Split 80/20
train_df, val_df = None, None  # YOUR CODE: train_test_split(final_df, test_size=0.2, random_state=42)

print(f"\n✓ Train/validation split complete")
print(f"  Training samples: {len(train_df)}")
print(f"  Validation samples: {len(val_df)}")

# Step 7: Save to local CSV (NO headers, NO index - XGBoost requirement)
train_path = '/tmp/train.csv'
val_path = '/tmp/validation.csv'

# YOUR CODE: Save DataFrames as CSV
# train_df.to_csv(train_path, header=False, index=False)
None  # YOUR CODE
None  # YOUR CODE: Same for val_df

print(f"\n✓ CSV files saved locally")

# Step 8: Upload to S3
train_s3_key = f"{OUTPUT_PREFIX}/train.csv"
val_s3_key = f"{OUTPUT_PREFIX}/validation.csv"

# YOUR CODE: Upload files to S3
# s3.upload_file(train_path, BUCKET_NAME, train_s3_key)
None  # YOUR CODE
None  # YOUR CODE: Same for validation

print(f"\n✓ Files uploaded to S3")
print(f"  Train:      s3://{BUCKET_NAME}/{train_s3_key}")
print(f"  Validation: s3://{BUCKET_NAME}/{val_s3_key}")

print("\n" + "=" * 60)
print("✅ Feature engineering complete!")
print("=" * 60)

In [ ]:
# =============================================================================
# VIEW FINAL TRAINING DATA
# =============================================================================

print("Final Training Data Preview:")
print("=" * 60)

# Show training data structure
print(f"Training set shape: {train_df.shape}")
print(f"Features: {train_df.shape[1] - 1} (plus 1 target column)")
print(f"\nColumn order (target must be first):")
for i, col in enumerate(train_df.columns):
    marker = "🎯 TARGET" if i == 0 else f"   Feature {i}"
    print(f"  {i}: {col:20s} {marker}")

# Show first few rows
print("\n" + "=" * 60)
print("First few training samples:")
print("=" * 60)
display(train_df.head())

# Verify S3 upload
print("\n" + "=" * 60)
print("Verifying S3 upload:")
print("=" * 60)

try:
    train_obj = s3.head_object(Bucket=BUCKET_NAME, Key=train_s3_key)
    val_obj = s3.head_object(Bucket=BUCKET_NAME, Key=val_s3_key)
    
    print(f"✓ train.csv:      {train_obj['ContentLength']} bytes")
    print(f"✓ validation.csv: {val_obj['ContentLength']} bytes")
    print(f"\n✅ Data ready for SageMaker training!")
except Exception as e:
    print(f"⚠️ Could not verify S3 upload: {e}")

# Show fraud distribution in splits
print("\n" + "=" * 60)
print("Fraud Distribution in Splits:")
print("=" * 60)
print(f"Training:   {train_df['is_fraud'].sum()} fraud / {len(train_df)} total")
print(f"Validation: {val_df['is_fraud'].sum()} fraud / {len(val_df)} total")

# Part 4: Training with SageMaker XGBoost

Now that we have our training data in S3, we're ready to train a machine learning model using **Amazon SageMaker**.

## SageMaker Training Components

| Component | Purpose | Our Configuration |
|-----------|---------|-------------------|
| **Estimator** | Defines the training job configuration | XGBoost built-in algorithm |
| **Instance Type** | Compute resources for training | `ml.m5.xlarge` (4 vCPU, 16 GB RAM) |
| **Hyperparameters** | Model configuration | Learning rate, depth, objective |
| **Input Data** | Training and validation data locations | S3 paths we just created |
| **Output** | Where to save the trained model | S3 bucket model artifacts |

## XGBoost for Fraud Detection

**XGBoost** (eXtreme Gradient Boosting) is ideal for fraud detection because:
- **Handles imbalanced data well** (fraud is rare)
- **Fast training** on tabular data
- **Feature importance** - shows which features matter most
- **Built-in algorithm** - no custom Docker containers needed

## Key Hyperparameters

```python
{
    'objective': 'binary:logistic',      # Binary classification (fraud yes/no)
    'num_round': 100,                    # Number of boosting rounds
    'max_depth': 5,                      # Tree depth (prevents overfitting)
    'eta': 0.2,                          # Learning rate
    'subsample': 0.8,                    # Fraction of samples per tree
    'scale_pos_weight': 1                # Handle imbalanced classes
}
```

## Demo: Training a Model

Let's train an XGBoost model on our call center fraud detection dataset. This is complete code - no lab for this section, just observe the training process.

In [ ]:
# =============================================================================
# TRAIN XGBOOST MODEL ON SAGEMAKER
# =============================================================================

print("Configuring SageMaker XGBoost training job...")
print("=" * 60)

# Get the XGBoost container image URI for our region
from sagemaker.image_uris import retrieve

xgboost_container = retrieve('xgboost', region, version='1.5-1')
print(f"XGBoost Container: {xgboost_container}")

# Configure the Estimator
xgb_estimator = Estimator(
    image_uri=xgboost_container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{BUCKET_NAME}/{OUTPUT_PREFIX}/models',
    sagemaker_session=sm_session,
    base_job_name='week6-fraud-xgboost'
)

# Set hyperparameters
xgb_estimator.set_hyperparameters(
    objective='binary:logistic',    # Binary classification
    num_round=100,                  # Number of boosting rounds
    max_depth=5,                    # Maximum tree depth
    eta=0.2,                        # Learning rate
    gamma=4,                        # Minimum loss reduction for split
    min_child_weight=6,             # Minimum sum of instance weight in child
    subsample=0.8,                  # Fraction of samples per tree
    colsample_bytree=0.8,           # Fraction of features per tree
    scale_pos_weight=1,             # Balance positive/negative weights
    eval_metric='auc',              # Use AUC for evaluation
    early_stopping_rounds=10        # Stop if no improvement
)

print("\n✓ Estimator configured")
print(f"  Instance: {xgb_estimator.instance_type}")
print(f"  Container: XGBoost 1.5-1")
print(f"  Output: s3://{BUCKET_NAME}/{OUTPUT_PREFIX}/models/")

# Define input data channels
train_input = TrainingInput(
    s3_data=f's3://{BUCKET_NAME}/{train_s3_key}',
    content_type='text/csv'
)

validation_input = TrainingInput(
    s3_data=f's3://{BUCKET_NAME}/{val_s3_key}',
    content_type='text/csv'
)

print("\n✓ Data channels configured")
print(f"  Training:   s3://{BUCKET_NAME}/{train_s3_key}")
print(f"  Validation: s3://{BUCKET_NAME}/{val_s3_key}")

# Start training job
print("\n" + "=" * 60)
print("🚀 Starting training job...")
print("=" * 60)
print("This will take 3-5 minutes. Progress will be shown below.\n")

xgb_estimator.fit(
    inputs={
        'train': train_input,
        'validation': validation_input
    },
    wait=True,     # Block until training completes
    logs='All'     # Stream all logs to notebook
)

print("\n" + "=" * 60)
print("✅ Training job completed!")
print("=" * 60)

In [ ]:
# =============================================================================
# VIEW TRAINING METRICS AND MODEL ARTIFACTS
# =============================================================================

print("Training Job Summary:")
print("=" * 60)

# Get job details
training_job_name = xgb_estimator.latest_training_job.name
job_details = sm_session.sagemaker_client.describe_training_job(
    TrainingJobName=training_job_name
)

# Show key metrics
print(f"Job Name:     {training_job_name}")
print(f"Status:       {job_details['TrainingJobStatus']}")
print(f"Instance:     {job_details['ResourceConfig']['InstanceType']}")
print(f"Training time: {job_details['TrainingTimeInSeconds']} seconds")
print(f"Billable time: {job_details['BillableTimeInSeconds']} seconds")

# Model artifacts
model_artifacts = job_details['ModelArtifacts']['S3ModelArtifacts']
print(f"\nModel Artifacts:")
print(f"  {model_artifacts}")

# Final metrics from training
if 'FinalMetricDataList' in job_details:
    print("\n" + "=" * 60)
    print("Final Metrics:")
    print("=" * 60)
    for metric in job_details['FinalMetricDataList']:
        print(f"  {metric['MetricName']:20s}: {metric['Value']:.4f}")

# Show training curve (if available in logs)
print("\n" + "=" * 60)
print("💡 Training Insights:")
print("=" * 60)
print("✓ Model trained on call transcript NLP features + transaction patterns")
print("✓ XGBoost learned to combine sentiment, entities, amounts, and frequency")
print("✓ Model artifacts saved to S3 and ready for deployment (Week 7!)")
print("\n🔗 View full training job details in SageMaker Console:")
print(f"   https://console.aws.amazon.com/sagemaker/home?region={region}#/jobs/{training_job_name}")

# Part 5: Hyperparameter Tuning

We just trained a model with **manually chosen hyperparameters**. But how do we know if these are optimal?

**Hyperparameter tuning** automates the search for the best hyperparameter configuration by training multiple models and comparing their performance.

## What is Hyperparameter Tuning?

| Concept | Description | Example |
|---------|-------------|---------|
| **Hyperparameter** | Configuration set before training | `max_depth`, `eta`, `subsample` |
| **Tuning Strategy** | How to search the space | Bayesian optimization (smart search) |
| **Objective Metric** | What to optimize | Maximize validation AUC |
| **Search Range** | Possible values to try | `max_depth`: 3 to 10, `eta`: 0.01 to 0.3 |

## Bayesian Optimization vs Grid Search

**Grid Search** (naive approach):
- Try every combination of parameters
- Very slow (hundreds of training jobs)
- Example: 5 values × 5 values × 5 values = 125 jobs

**Bayesian Optimization** (smart approach):
- Learn from previous results
- Focus search on promising regions
- Typically finds good results in 20-50 jobs

## SageMaker HyperparameterTuner

```python
from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter

tuner = HyperparameterTuner(
    estimator=xgb_estimator,
    objective_metric_name='validation:auc',
    objective_type='Maximize',
    max_jobs=20,                      # Total training jobs to run
    max_parallel_jobs=2,              # Jobs to run simultaneously
    hyperparameter_ranges={
        'max_depth': IntegerParameter(3, 10),
        'eta': ContinuousParameter(0.01, 0.3),
        'subsample': ContinuousParameter(0.5, 0.9)
    }
)
```

## Demo: Launch a Tuning Job

Let's launch a tuning job that will run in the background. We won't wait for it to complete (it takes 30-60 minutes), but you'll see how to monitor progress.

In [ ]:
# =============================================================================
# LAUNCH HYPERPARAMETER TUNING JOB
# =============================================================================

print("Configuring hyperparameter tuning job...")
print("=" * 60)

# Create a new estimator for tuning (clone the previous one)
xgb_tuning_estimator = Estimator(
    image_uri=xgboost_container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{BUCKET_NAME}/{OUTPUT_PREFIX}/tuning',
    sagemaker_session=sm_session,
    base_job_name='week6-fraud-tuning'
)

# Set static hyperparameters (ones we won't tune)
xgb_tuning_estimator.set_hyperparameters(
    objective='binary:logistic',
    num_round=100,
    eval_metric='auc',
    early_stopping_rounds=10
)

# Define hyperparameter ranges to explore
hyperparameter_ranges = {
    'max_depth': IntegerParameter(3, 10),           # Tree depth
    'eta': ContinuousParameter(0.01, 0.3),          # Learning rate
    'subsample': ContinuousParameter(0.5, 0.9),     # Sample fraction
    'colsample_bytree': ContinuousParameter(0.5, 0.9),  # Feature fraction
    'min_child_weight': IntegerParameter(1, 10),    # Minimum child weight
    'gamma': ContinuousParameter(0, 5)              # Min loss reduction
}

print("✓ Hyperparameter ranges defined:")
for param, range_obj in hyperparameter_ranges.items():
    print(f"  {param:20s}: {range_obj}")

# Create tuner
tuner = HyperparameterTuner(
    estimator=xgb_tuning_estimator,
    objective_metric_name='validation:auc',
    objective_type='Maximize',
    max_jobs=20,                # Total training jobs to try
    max_parallel_jobs=2,        # Run 2 jobs at a time
    hyperparameter_ranges=hyperparameter_ranges,
    strategy='Bayesian',        # Smart search strategy
    base_tuning_job_name='week6-fraud-tuning'
)

print("\n✓ Tuner configured")
print(f"  Strategy:      Bayesian optimization")
print(f"  Max jobs:      20 training jobs")
print(f"  Parallel jobs: 2 simultaneous")
print(f"  Objective:     Maximize validation:auc")

# Launch tuning job (runs in background)
print("\n" + "=" * 60)
print("🚀 Launching hyperparameter tuning job...")
print("=" * 60)

tuner.fit(
    inputs={
        'train': train_input,
        'validation': validation_input
    },
    wait=False,     # Don't block - job runs in background
    logs=False      # Don't stream logs (would be overwhelming with 20 jobs)
)

tuning_job_name = tuner.latest_tuning_job.name

print(f"\n✅ Tuning job launched: {tuning_job_name}")
print(f"\n⏱️ This job will take approximately 30-60 minutes to complete")
print(f"   (20 training jobs × 3-5 min each ÷ 2 parallel = ~30-50 min)")
print(f"\n🔗 View progress in SageMaker Console:")
print(f"   https://console.aws.amazon.com/sagemaker/home?region={region}#/hyper-tuning-jobs/{tuning_job_name}")
print(f"\n💡 The job runs asynchronously - you can continue with the next section")

In [ ]:
# =============================================================================
# MONITOR TUNING JOB STATUS
# =============================================================================

print("Checking tuning job status...")
print("=" * 60)

# Get current status
tuning_job_details = sm_session.sagemaker_client.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuning_job_name
)

status = tuning_job_details['HyperParameterTuningJobStatus']
print(f"Status: {status}")

# Show progress
if 'TrainingJobStatusCounters' in tuning_job_details:
    counters = tuning_job_details['TrainingJobStatusCounters']
    print(f"\nTraining Job Progress:")
    print(f"  Completed:   {counters.get('Completed', 0)}")
    print(f"  InProgress:  {counters.get('InProgress', 0)}")
    print(f"  Stopped:     {counters.get('Stopped', 0)}")
    print(f"  Failed:      {counters.get('RetryableError', 0) + counters.get('NonRetryableError', 0)}")

# Show best result so far (if any jobs completed)
if 'BestTrainingJob' in tuning_job_details:
    best_job = tuning_job_details['BestTrainingJob']
    print(f"\n✅ Best Training Job So Far:")
    print(f"  Job Name: {best_job['TrainingJobName']}")
    print(f"  Objective: {best_job['FinalHyperParameterTuningJobObjectiveMetric']['Value']:.4f}")
    print(f"\n  Best Hyperparameters:")
    for param, value in best_job['TunedHyperParameters'].items():
        print(f"    {param:20s}: {value}")
else:
    print(f"\n⏳ No completed jobs yet - check back in a few minutes")

# Provide code for later checking
print("\n" + "=" * 60)
print("💡 To check status later, run this code:")
print("=" * 60)
print(f"""
import boto3
sm_client = boto3.client('sagemaker', region_name='{region}')
details = sm_client.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName='{tuning_job_name}'
)
print(f"Status: {{details['HyperParameterTuningJobStatus']}}")
if 'BestTrainingJob' in details:
    best = details['BestTrainingJob']
    print(f"Best AUC: {{best['FinalHyperParameterTuningJobObjectiveMetric']['Value']:.4f}}")
""")

print("\n" + "=" * 60)
print("🎯 What's Happening:")
print("=" * 60)
print("1. Bayesian optimizer selects hyperparameter combinations to try")
print("2. Each combination trains an XGBoost model (3-5 minutes)")
print("3. Two models train in parallel to save time")
print("4. After each job, the optimizer learns and picks better combinations")
print("5. After 20 jobs, the best hyperparameters are identified")
print("\n✓ The optimized model will outperform our manually tuned one!")

# Summary: What We Built Today

## The Complete Pipeline

We built an **end-to-end intelligent call center fraud detection system**:

```
Call Audio (WAV files)
         ↓
    [Amazon Transcribe]
    ✓ Async transcription jobs
    ✓ Speaker diarization
    ✓ Timeout handling with fallback
         ↓
    Text Transcripts
         ↓
    [Amazon Comprehend]
    ✓ Sentiment analysis
    ✓ Entity extraction
    ✓ Key phrase detection
         ↓
    NLP Features
         ↓
    [Feature Engineering]
    ✓ Combined NLP + transaction data
    ✓ Created ML-ready features
    ✓ Prepared XGBoost format (target first, no headers)
         ↓
    Training Data in S3
         ↓
    [SageMaker XGBoost Training]
    ✓ Trained binary classification model
    ✓ Used built-in XGBoost algorithm
    ✓ Evaluated on validation set
         ↓
    Trained Model
         ↓
    [Hyperparameter Tuning]
    ✓ Automated search for optimal config
    ✓ Bayesian optimization strategy
    ✓ 20 training jobs, 2 parallel
         ↓
    Optimized Fraud Detection Model
```

## AWS Services Used

| Service | Purpose | Key APIs Used |
|---------|---------|---------------|
| **Amazon Transcribe** | Audio → Text | `start_transcription_job()`, `get_transcription_job()` |
| **Amazon Comprehend** | NLP Analysis | `detect_sentiment()`, `detect_entities()`, `detect_key_phrases()` |
| **Amazon S3** | Data storage | Store audio, transcripts, training data, models |
| **Amazon SageMaker** | ML Training | `Estimator.fit()`, `HyperparameterTuner.fit()` |
| **XGBoost** | Algorithm | Built-in binary classification |

## Key Takeaways

### 1. **Async Job Pattern**
- Transcribe jobs are asynchronous - poll for completion
- Always implement timeouts and fallback strategies
- Use unique job names (include timestamps)

### 2. **Feature Engineering**
- Combined structured data (transactions) with unstructured data (call transcripts)
- NLP features (sentiment, entities) capture emotional context
- Transaction features (amounts, frequency) capture behavioral patterns

### 3. **SageMaker Training**
- Built-in algorithms (XGBoost) simplify ML deployment
- Estimator API provides consistent training interface
- Models automatically save to S3

### 4. **Hyperparameter Tuning**
- Automates optimization of model configuration
- Bayesian optimization is smarter than grid search
- Background jobs allow you to continue working

## Real-World Impact

This pipeline can:
- **Reduce fraud investigation time** by 70% through automated prioritization
- **Scale to thousands of calls daily** without manual review
- **Identify complex patterns** humans might miss (e.g., sentiment + transaction timing)
- **Provide explainable insights** through entity extraction and sentiment scores

# Optional / Extra Labs

Finished early? Want to explore further? Try these advanced challenges!

## Extra Lab 1: Batch Transcription

**Challenge**: Modify the transcription code to use **batch transcription** instead of individual jobs.

**Why**: Batch jobs are more efficient for large volumes and can process multiple files in one operation.

**Hints**:
- Look into `start_call_analytics_job()` API
- Can analyze multiple speakers and sentiment in one API call
- Combines transcription + some NLP analysis

## Extra Lab 2: Custom Vocabulary

**Challenge**: Add a **custom vocabulary** to Transcribe to improve accuracy for financial terms.

**Why**: Generic transcription may mis-transcribe domain-specific words like "chargeback", "ACH", "wire transfer".

**Hints**:
- Create custom vocabulary with financial terms
- Use `create_vocabulary()` API
- Reference vocabulary in `start_transcription_job()` settings
- Financial terms to include: "chargeback", "ACH transfer", "wire transfer", "fraud alert"

## Extra Lab 3: Spanish Call Translation

**Challenge**: Implement full translation workflow for call_005 (Spanish).

**Why**: Real call centers serve multilingual customers.

**Steps**:
1. Use `detect_dominant_language()` on transcript
2. If not English, use `translate_text()` to translate
3. Analyze the translated text with Comprehend
4. Compare results to original English calls

**Hint**: `translate_text(Text=text, SourceLanguageCode='es', TargetLanguageCode='en')`

## Extra Lab 4: LSTM Sentiment Model

**Challenge**: Train a **custom LSTM model** for sentiment analysis instead of using Comprehend.

**Why**: Custom models can be fine-tuned for domain-specific language (financial call center context).

**Hints**:
- Use PyTorch or TensorFlow
- Bring Your Own Container (BYOC) to SageMaker
- Compare results to Comprehend's generic sentiment
- Requires labeled sentiment data (create synthetic or manual labels)

## Extra Lab 5: Deploy the Model

**Challenge**: Deploy the trained XGBoost model to a **SageMaker endpoint** for real-time predictions.

**Why**: Week 7 will cover deployment - get a head start!

**Hints**:
- Use `xgb_estimator.deploy()`
- Create endpoint with `initial_instance_count=1`
- Test with sample features
- Remember to delete endpoint after testing (cost control!)

## Extra Lab 6: Feature Importance Analysis

**Challenge**: Extract and visualize **feature importance** from the XGBoost model.

**Why**: Understanding which features matter most helps refine the pipeline and explain predictions.

**Hints**:
- Download model artifacts from S3
- Load with `xgboost.Booster()`
- Use `get_score(importance_type='gain')`
- Create bar chart showing top features
- Are NLP features or transaction features more important?

## Async Activity: Cost Analysis

While the tuning job runs, calculate the **cost of this pipeline**:

1. **Transcribe**: $0.0004/second × 5 calls × 60 seconds = ?
2. **Comprehend**: $0.0001/request × 5 calls × 3 APIs = ?
3. **SageMaker Training**: ml.m5.xlarge × $0.23/hour × 0.1 hours = ?
4. **Hyperparameter Tuning**: ml.m5.xlarge × $0.23/hour × 20 jobs × 0.08 hours = ?
5. **S3 Storage**: Negligible (<1 GB)

**Total pipeline cost**: Estimate and compare to manual fraud analyst review (assume $50/hour × 2 hours per case)

# Congratulations! 🎉

You've successfully built an **end-to-end intelligent ML pipeline** combining multiple AWS AI services!

## What You Accomplished Today

✅ **Transcribed call recordings** with Amazon Transcribe (async job pattern)  
✅ **Analyzed sentiment and entities** with Amazon Comprehend  
✅ **Engineered features** combining NLP insights with transaction data  
✅ **Trained an XGBoost model** on SageMaker using the Estimator API  
✅ **Launched hyperparameter tuning** to optimize model performance

## Skills You Practiced

### AWS Services Integration
- Connected multiple AWS services in a cohesive pipeline
- Managed async jobs with polling and timeouts
- Stored and retrieved data from S3

### Machine Learning
- Feature engineering from unstructured text data
- Binary classification for fraud detection
- Hyperparameter optimization with Bayesian search

### Production Patterns
- Fallback strategies for reliability (pre-computed transcripts)
- Data format requirements (XGBoost CSV format)
- Background job management (tuning runs async)

## Next Week: Week 7 - Deployment and Monitoring

In **Week 7**, we'll take this trained model into production:

### What's Coming
- **Deploy models to endpoints** for real-time inference
- **Create batch inference jobs** for large-scale predictions
- **Track experiments with MLflow** (versioning, metrics, artifacts)
- **Monitor model performance** (drift detection, retraining triggers)
- **Cost optimization** (auto-scaling, serverless endpoints)

### Preview: From Training to Production
```
Week 6: Built and trained model
         ↓
Week 7: Deploy model → SageMaker Endpoint
         ↓
       Make predictions on new calls
         ↓
       Monitor accuracy and drift
         ↓
       Retrain when performance degrades
```

## Resources

- **SageMaker Documentation**: [https://docs.aws.amazon.com/sagemaker/](https://docs.aws.amazon.com/sagemaker/)
- **XGBoost Algorithm Guide**: [Built-in Algorithm Reference](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html)
- **Hyperparameter Tuning Best Practices**: [Tuning Guide](https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning.html)
- **AWS AI Services**: [Transcribe](https://docs.aws.amazon.com/transcribe/) | [Comprehend](https://docs.aws.amazon.com/comprehend/)

## Clean Up

**IMPORTANT - Cost Control**:
- ✅ Training jobs automatically stop after completion
- ✅ Hyperparameter tuning jobs automatically stop after 20 jobs
- ⚠️ **Week 7**: Remember to delete endpoints after testing (endpoints run continuously and cost ~$0.30/hour)
- ✅ S3 storage costs are negligible (<$0.01/month for this dataset)

**Saturday Cleanup**: All SageMaker apps will be automatically shut down at 00:00 Buenos Aires time (after Friday classes).

Great work today! See you in Week 7! 🚀